# AD vs FTD, stage 2: an AD-vs-FTD specialist on covariance and connectivity matrices

Stage 2 of the two-stage plan. It is trained on dementia patients only and outputs
`q_AD = P(AD | dementia)` for **every** patient. The folds are the 3-class notebook's patient folds, so a stage-1
dementia-vs-HC model can later be combined with it patient by patient.

- **Design fixed in Cell 2** (`results/riemann/protocol.json`) before any matrix is computed.
- **Candidates:** band power (the floor); Riemannian covariance; lagged cross-correlation
  (Mlinarič et al. 2026); a parameterised spectrum; a soft vote of the last three. Each is tuned and
  the best is picked by inner cross-validation on the outer training patients only. Every candidate is reported.
- **Reads** `../data/derivatives/`, `features/` (band power, spectral features), `results/aperiodic/subject_scores.csv`
  and `../subject_cache/` (labels only, to rebuild the folds).
- **Writes** `features/riemann_matrices.npz` and `results/riemann/`.
- About 5 min, plus ~20 min for the permutation test in Cell 9 (CPU).

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ===========================================================================
import json, os, time, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from pyriemann.tangentspace import TangentSpace
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

EXP_DIR   = Path.cwd()                        # experiments_ad_vs_ftd/
PROJECT   = EXP_DIR.parent
DATA_ROOT = PROJECT / "data"                  # ds004504 with derivatives/
CACHE_DIR = PROJECT / "subject_cache"         # labels only: rebuilds the 3-class notebook's folds
FEAT_DIR  = EXP_DIR / "features"
OUT_DIR   = EXP_DIR / "results" / "riemann"
assert (DATA_ROOT / "participants.tsv").exists(), \
    f"no dataset at {DATA_ROOT}: open this notebook from experiments_ad_vs_ftd/"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- the data every subject contributes: identical to features_ad_ftd.ipynb ----
SFREQ    = 500.0
EPOCH_S  = 4.0               # non-overlapping epochs on a fixed 4 s grid from t = 0
N_EPOCHS = 75                # 75 x 4 s = 300 s; the shortest recording has exactly 75 clean epochs
EPOCH_N  = int(EPOCH_S * SFREQ)
CHANNELS = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
            "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]          # subject_cache order
BANDS = {"delta": (0.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0),
         "beta": (13.0, 30.0)}   # gamma left out: at the scalp it is mostly muscle
HC, AD, FTD = 0, 1, 2

# ---- matrices: fixed here, before any matrix is computed -------------------
LOADING   = 0.01     # covariance + 0.01 x mean variance x I: ICA left the cleaned data with rank 16-19
DECIM     = 4        # cross-correlation on 125 Hz copies of each band (every band ends at <= 30 Hz)
MAX_LAG   = 12       # samples at 125 Hz = +-96 ms: scalp conduction delays, less than half a theta cycle

# ---- protocol --------------------------------------------------------------
SEED, N_FOLDS, N_REPEATS = 42, 5, 5   # the 3-class notebook's folds; its repeats 1-3 are repeats 1-3 here
INNER_FOLDS = 5
C_GRID      = [0.001, 0.01, 0.1, 1.0]
N_BOOT      = 2000
N_PERM      = 100                     # label shuffles of the whole nested pipeline, repeat 1 (Cell 9)
N_JOBS      = max(1, min(8, (os.cpu_count() or 2) - 2))
FORCE_MATRICES = False                # True = recompute features/riemann_matrices.npz

print(f"data    : {DATA_ROOT}\noutput  : {OUT_DIR}\nworkers : {N_JOBS}")


data    : C:\Projects\Major Project\data
output  : C:\Projects\Major Project\experiments_ad_vs_ftd\results\riemann
workers : 8


In [2]:
# ===========================================================================
# CELL 2 — PROTOCOL, WRITTEN BEFORE ANY MATRIX IS COMPUTED
# ===========================================================================
PROTOCOL = {
    "question": "Stage 2 of the two-stage plan: AD (36) vs FTD (23), trained on dementia patients only. "
                "q_AD = P(AD | dementia) is predicted for every test patient, healthy controls included, "
                "so that a stage-1 dementia-vs-HC model can be combined with it",
    "data": "first 75 join-free 4 s epochs per subject (300 s), as features_ad_ftd.ipynb; "
            "common average reference for C2 and C3",
    "candidates": {
        "C1 band power": "the cache's 95 band powers (features/eeg_features.csv, bp_*) -> standardise -> "
                         "L2 logistic regression",
        "C2 Riemannian covariance": "per band (delta 0.5-4, theta 4-8, alpha 8-13, beta 13-30 Hz, zero-phase FIR): "
                                    "covariance of the 300 s + 0.01 x mean variance x I; tangent space at the "
                                    "training patients' Riemannian mean; 4 x 190 values -> standardise -> L2 "
                                    "logistic regression",
        "C3 lagged cross-correlation": "per band: normalised cross-correlation per epoch, averaged over the 75 epochs "
                                       "at each lag up to +-96 ms (125 Hz); per channel pair the value at the lag "
                                       "with the largest |r|, sign kept (Mlinaric et al. 2026, Sci Rep 16:4903); "
                                       "Fisher z of the 171 pairs, 4 x 171 values -> standardise -> L2 logistic "
                                       "regression. Vectorised, not Riemannian: the cleaned data are rank-deficient",
        "C4 parameterised spectrum": "aperiodic exponent and offset (Experiment 8, 2 s windows), posterior alpha "
                                     "centre of gravity, log theta/alpha in 5 regions; 8 values -> standardise -> "
                                     "L2 logistic regression",
        "C5 soft vote": "mean of the C2, C3 and C4 probabilities, each with its own tuned C",
    },
    "tuning": "C in [0.001, 0.01, 0.1, 1] by 5-fold inner CV AUC on the outer training dementia patients; "
              "balanced class weights",
    "selection": "per outer fold, the candidate with the highest nested inner-CV AUC (its tuning included) is the "
                 "selected pipeline; its outer prediction is the headline. Every candidate's outer result is "
                 "reported too",
    "folds": "outer = GAT_multiclass_ds004504.ipynb make_splits: StratifiedGroupKFold over the cache's windows "
             "grouped by subject, seed 42 + 1000 r, 5 repeats",
    "metrics": "AD vs FTD on the 59 dementia patients: AUC of q_AD, balanced accuracy at 0.5, recall AD and FTD; "
               "pooled per repeat, mean over repeats; 95 % CI from 2000 stratified subject bootstraps of the "
               "repeat mean",
    "tests": "paired bootstrap of AUC vs C1; Holm over C2-C5; the selected pipeline vs C1 reported separately; "
             "chance level from 100 label shuffles of the whole nested pipeline on repeat 1",
    "confounds": "likelihood-ratio test of each score added to an unpenalised logistic model of MMSE + age + sex; "
                 "AUC of sex, age, MMSE and ICA rank deficit alone",
    "target": "AD vs FTD AUC >= 0.70 (best honest published: 0.651, Mlinaric et al. 2026, LOSO)",
    "decision": "the result is reported whatever it is",
}

_path = OUT_DIR / "protocol.json"
if _path.exists():                    # a re-run keeps the original date and must not change the design
    _old = json.loads(_path.read_text(encoding="utf-8"))
    assert {k: v for k, v in _old.items() if k != "written"} == PROTOCOL, "protocol changed after it was written"
    PROTOCOL = _old
else:
    PROTOCOL = {"written": time.strftime("%Y-%m-%d %H:%M") + ", before any matrix was computed", **PROTOCOL}
    _path.write_text(json.dumps(PROTOCOL, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"protocol: {_path}  (written {PROTOCOL['written']})")


protocol: C:\Projects\Major Project\experiments_ad_vs_ftd\results\riemann\protocol.json  (written 2026-11-16 16:04, before any matrix was computed)


In [3]:
# ===========================================================================
# CELL 3 — SUBJECTS, CLEAN EPOCHS AND THE 3-CLASS NOTEBOOK'S FOLDS
# ===========================================================================
subjects = pd.read_csv(FEAT_DIR / "subjects.csv")
assert list(subjects.participant_id) == [f"sub-{i:03d}" for i in range(1, 89)]
LABELS   = subjects.label.to_numpy()
DEMENTIA = np.flatnonzero(LABELS != HC)                      # the 59 patients stage 2 learns from
print(f"{len(subjects)} subjects: " + ", ".join(f"{g} {n}" for g, n in subjects.group.value_counts().items()))


def set_file(sid):
    path = DATA_ROOT / "derivatives" / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def clean_epoch_starts(raw):
    """Start sample of every 4 s grid epoch with no ASR join ('boundary') inside, and the join count."""
    joins = np.array([a["onset"] for a in raw.annotations if a["description"] == "boundary"]) * SFREQ
    starts = np.arange(0, raw.n_times - EPOCH_N + 1, EPOCH_N)
    inside = (joins[None, :] > starts[:, None]) & (joins[None, :] < starts[:, None] + EPOCH_N)
    return starts[~inside.any(axis=1)], len(joins)


def make_folds():
    """GAT_multiclass_ds004504.ipynb's make_splits, rebuilt from the cache's window labels and mapped to subjects."""
    Y, SIDW = [], []
    for sid, lab in zip(subjects.participant_id, LABELS):
        with np.load(CACHE_DIR / f"{sid}.npz") as z:
            y = z["y"]
        assert (y == lab).all(), f"{sid}: cache label differs from subjects.csv"
        Y.append(y); SIDW.append(np.full(len(y), sid))
    Y, SIDW = np.concatenate(Y), np.concatenate(SIDW)
    folds = []
    for r in range(1, N_REPEATS + 1):
        outer = StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED + 1000 * r)
        for f, (_, te) in enumerate(outer.split(np.arange(len(Y)), Y, groups=SIDW), start=1):
            test = np.flatnonzero(subjects.participant_id.isin(np.unique(SIDW[te])).to_numpy())
            folds.append(dict(repeat=r, fold=f, test=test, n_windows=len(te)))
    return folds


FOLDS = make_folds()
_lines = [f"r{s['repeat']} f{s['fold']}: test {s['n_windows']:4d}w/{len(s['test']):2d}s ("
          + "/".join(str((LABELS[s['test']] == c).sum()) for c in (HC, AD, FTD)) + ")" for s in FOLDS]
for r in range(1, N_REPEATS + 1):
    tested = np.concatenate([s["test"] for s in FOLDS if s["repeat"] == r])
    assert np.array_equal(np.sort(tested), np.arange(len(subjects))), f"repeat {r} does not test everyone once"

# the 3-class notebook printed its folds; repeats 1-3 must match them line for line
_nb = json.loads((PROJECT / "GAT_multiclass_ds004504.ipynb").read_text(encoding="utf-8"))
_printed = "".join("".join(o.get("text", "")) for o in _nb["cells"][4].get("outputs", []))
_expected = [ln.split(": train")[0] + ": " + ln.split("| ")[1].split("   [")[0]
             for ln in _printed.splitlines() if ln.startswith("r") and "| test" in ln]
assert _expected and _lines[:len(_expected)] == _expected, "folds differ from GAT_multiclass_ds004504.ipynb"
print("\n".join(_lines))
print(f"\neach repeat tests all 88 subjects once; repeats 1-{len(_expected) // N_FOLDS} match the 3-class "
      f"notebook's printed folds exactly   [test: windows/subjects (HC/AD/FTD)]")


88 subjects: AD 36, HC 29, FTD 23


r1 f1: test 1660w/18s (6/7/5)
r1 f2: test 1652w/18s (6/7/5)
r1 f3: test 1618w/18s (6/8/4)
r1 f4: test 1560w/17s (5/7/5)
r1 f5: test 1560w/17s (6/7/4)
r2 f1: test 1660w/18s (6/7/5)
r2 f2: test 1652w/18s (6/7/5)
r2 f3: test 1618w/18s (6/8/4)
r2 f4: test 1560w/17s (5/7/5)
r2 f5: test 1560w/17s (6/7/4)
r3 f1: test 1652w/18s (6/7/5)
r3 f2: test 1618w/18s (6/8/4)
r3 f3: test 1560w/17s (6/7/4)
r3 f4: test 1660w/18s (6/7/5)
r3 f5: test 1560w/17s (5/7/5)
r4 f1: test 1652w/18s (6/7/5)
r4 f2: test 1618w/18s (5/8/5)
r4 f3: test 1560w/17s (6/7/4)
r4 f4: test 1660w/18s (6/7/5)
r4 f5: test 1560w/17s (6/7/4)
r5 f1: test 1652w/18s (6/7/5)
r5 f2: test 1618w/18s (6/8/4)
r5 f3: test 1560w/17s (6/7/4)
r5 f4: test 1660w/18s (6/7/5)
r5 f5: test 1560w/17s (5/7/5)

each repeat tests all 88 subjects once; repeats 1-3 match the 3-class notebook's printed folds exactly   [test: windows/subjects (HC/AD/FTD)]


In [4]:
# ===========================================================================
# CELL 4 — MATRICES: COVARIANCE AND LAGGED CROSS-CORRELATION (self-tests first)
# ===========================================================================
IU = np.triu_indices(len(CHANNELS), 1)                       # the 171 channel pairs


def loaded_cov(x):
    """x (channels, samples), zero mean. Covariance + LOADING x mean variance on the diagonal."""
    c = x @ x.T / x.shape[1]
    return c + LOADING * np.trace(c) / len(c) * np.eye(len(c))


def lagged_xcorr(ep, max_lag=MAX_LAG):
    """ep (epochs, channels, samples). Normalised cross-correlation per epoch, averaged over epochs, at every
    lag in -max_lag..max_lag. Per pair: the value at the lag with the largest |r| (sign kept), and that lag.
    R[l, i, j] = corr(x_i(t + l), x_j(t)); a negative lag is the transpose of the positive one."""
    ep = ep - ep.mean(-1, keepdims=True)
    n = ep.shape[-1]
    norm = np.sqrt((ep ** 2).sum(-1))                                        # (E, C)
    R = np.stack([(np.einsum("eit,ejt->eij", ep[:, :, l:], ep[:, :, :n - l])
                   / (norm[:, :, None] * norm[:, None, :])).mean(0) for l in range(max_lag + 1)])
    full = np.concatenate([R[:0:-1].transpose(0, 2, 1), R])                 # lags -max_lag .. +max_lag
    k = np.abs(full).argmax(0)
    return np.take_along_axis(full, k[None], 0)[0], k - max_lag


# ---- self-tests on signals whose answer is known ----------------------------
_rng = np.random.default_rng(0)
_s = _rng.standard_normal((75, 520))
_ep = _rng.standard_normal((75, 19, 500))
_ep[:, 0], _ep[:, 1], _ep[:, 2] = _s[:, 10:510], _s[:, 7:507], -_s[:, 10:510]   # 1 = 0 delayed 3; 2 = -0
_r, _lag = lagged_xcorr(_ep)
assert _r[0, 1] > 0.99 and abs(_lag[0, 1]) == 3 and _lag[0, 1] == -_lag[1, 0], "a delayed copy must give r = 1 at 3 samples"
assert _r[0, 2] < -0.99 and _lag[0, 2] == 0, "an inverted copy must give r = -1 at lag 0 (sign kept)"
assert np.abs(_r[3:, 3:][np.triu_indices(16, 1)]).max() < 0.1, "independent noise must give |r| < 0.1"
assert np.allclose(_r, _r.T) and np.allclose(np.diag(_r), 1), "the matrix must be symmetric with a unit diagonal"
_x = _rng.standard_normal((16, 19 * 3000)).T @ _rng.standard_normal((16, 19))     # rank 16, like the cleaned data
_c = loaded_cov((_x - _x.mean(1, keepdims=True)).T)                                # + average reference
assert np.linalg.eigvalsh(_c).min() > 0.005 * np.trace(_c) / 19, "loading must make a rank-16 covariance positive definite"
print("self-tests passed: delayed copy, inverted copy, independent noise, symmetry, loading of a rank-deficient covariance")


def subject_matrices(sid):
    with warnings.catch_warnings():                       # parallel workers do not inherit Cell 1's settings
        warnings.filterwarnings("ignore")
        raw = mne.io.read_raw_eeglab(set_file(sid), preload=False, verbose="ERROR")
    starts = clean_epoch_starts(raw)[0][:N_EPOCHS]
    x = raw.get_data(picks=CHANNELS, start=0, stop=int(starts[-1]) + EPOCH_N) * 1e6      # µV, A1-A2 reference
    ev = np.linalg.eigvalsh(np.cov(x))
    rank = int((ev > ev.max() * 1e-10).sum())             # 19 minus the dimensions ICA removed
    x = x - x.mean(0, keepdims=True)                      # common average reference
    covs, xcz, lags = [], [], []
    for lo, hi in BANDS.values():
        xb = mne.filter.filter_data(x, SFREQ, lo, hi, verbose="ERROR")    # zero-phase FIR on the continuous signal
        ep = np.stack([xb[:, s:s + EPOCH_N] for s in starts])           # (75, 19, 2000)
        covs.append(loaded_cov(np.concatenate(list(ep), axis=-1)))
        r, lag = lagged_xcorr(ep[:, :, ::DECIM])
        xcz.append(np.arctanh(np.clip(r[IU], -0.999, 0.999)))
        lags.append(lag[IU])
    return np.stack(covs), np.concatenate(xcz), np.stack(lags), rank


_mat = FEAT_DIR / "riemann_matrices.npz"
if _mat.exists() and not FORCE_MATRICES:
    with np.load(_mat) as z:
        COV, XCZ, LAGS, RANK = z["cov"], z["xcz"], z["lags"], z["rank"]
    print(f"loaded {_mat.name}")
else:
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS)(delayed(subject_matrices)(s) for s in subjects.participant_id)
    COV, XCZ, LAGS, RANK = (np.stack([r[k] for r in res]) for k in range(4))
    np.savez_compressed(_mat, cov=COV, xcz=XCZ, lags=LAGS, rank=RANK, subjects=subjects.participant_id.to_numpy(),
                        bands=np.array(list(BANDS)))
    print(f"computed {len(res)} subjects in {time.time() - t0:.0f} s -> {_mat.name}")

assert all(np.linalg.eigvalsh(c).min() > 0 for c in COV.reshape(-1, 19, 19)), "every covariance must be SPD"
assert np.isfinite(XCZ).all()
print(f"covariance {COV.shape}, cross-correlation {XCZ.shape}; ICA rank per group:")
print(pd.crosstab(subjects.group, RANK, colnames=["rank"]))
print(f"lag of the kept |r| (ms), median |lag| per band: "
      + ", ".join(f"{b} {np.median(np.abs(LAGS[:, k])) * DECIM / SFREQ * 1000:.0f}" for k, b in enumerate(BANDS)))


self-tests passed: delayed copy, inverted copy, independent noise, symmetry, loading of a rank-deficient covariance


computed 88 subjects in 41 s -> riemann_matrices.npz
covariance (88, 4, 19, 19), cross-correlation (88, 684); ICA rank per group:
rank   16  17  18  19
group                
AD      6  17  10   3
FTD     4   8   7   4
HC      2  15  12   0
lag of the kept |r| (ms), median |lag| per band: delta 4, theta 0, alpha 0, beta 0


In [5]:
# ===========================================================================
# CELL 5 — THE FOUR FEATURE TABLES
# ===========================================================================
_feat = pd.read_csv(FEAT_DIR / "eeg_features.csv").set_index("participant_id").loc[subjects.participant_id]
_aper = (pd.read_csv(EXP_DIR / "results" / "aperiodic" / "subject_scores.csv")
           .set_index("participant_id").loc[subjects.participant_id])
SPEC_COLS = ["spec_iaf_cog"] + [f"spec_ratio_theta_alpha_{r}" for r in
                                ("frontal", "central", "temporal", "parietal", "occipital")]
F = {"bp":   _feat[[c for c in _feat.columns if c.startswith("bp_")]].to_numpy(),
     "cov":  COV,
     "xcz":  XCZ,
     "spec": np.column_stack([_aper[["exponent_2s", "offset_2s"]].to_numpy(), _feat[SPEC_COLS].to_numpy()])}
for k, v in F.items():
    assert len(v) == len(subjects) and np.isfinite(v).all(), k
print("\n".join(f"{k:5s} {v.shape}" for k, v in F.items()))


bp    (88, 95)
cov   (88, 4, 19, 19)
xcz   (88, 684)
spec  (88, 8)


In [6]:
# ===========================================================================
# CELL 6 — CANDIDATES AND THE NESTED CROSS-VALIDATION
# ===========================================================================
class Take(BaseEstimator, TransformerMixin):
    """X holds subject indices; returns those subjects' rows of `data`. Lets every candidate take the same X."""
    def __init__(self, data=None):
        self.data = data

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return self.data[np.asarray(X).ravel().astype(int)]


class BandTangent(BaseEstimator, TransformerMixin):
    """One tangent space per band, at the training subjects' Riemannian mean; the bands are concatenated."""
    def __init__(self, metric="riemann"):
        self.metric = metric

    def fit(self, X, y=None):
        self.ts_ = [TangentSpace(metric=self.metric).fit(X[:, b]) for b in range(X.shape[1])]
        return self

    def transform(self, X):
        return np.hstack([ts.transform(X[:, b]) for b, ts in enumerate(self.ts_)])


def logreg():
    return LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)


def candidates(inner):
    tune = lambda pipe: GridSearchCV(pipe, {"logisticregression__C": C_GRID}, cv=inner, scoring="roc_auc")
    c2 = make_pipeline(Take(F["cov"]), BandTangent(), StandardScaler(), logreg())
    c3 = make_pipeline(Take(F["xcz"]), StandardScaler(), logreg())
    c4 = make_pipeline(Take(F["spec"]), StandardScaler(), logreg())
    return {"C1 band power":               tune(make_pipeline(Take(F["bp"]), StandardScaler(), logreg())),
            "C2 Riemannian covariance":    tune(c2),
            "C3 lagged cross-correlation": tune(c3),
            "C4 parameterised spectrum":   tune(c4),
            "C5 soft vote":                VotingClassifier([("cov", tune(c2)), ("xcorr", tune(c3)), ("spec", tune(c4))],
                                                            voting="soft")}


CANDIDATES = list(candidates(None))


def run_fold(test, labels, seed):
    """Train on the dementia patients outside `test`; return, per candidate, its nested inner-CV AUC (tuning
    included) and its q_AD for every test patient. Nothing here sees a test patient's label."""
    train = np.setdiff1d(np.flatnonzero(labels != HC), test)
    X, y = train.reshape(-1, 1), (labels[train] == AD).astype(int)
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)
    out = {}
    for name, est in candidates(inner).items():
        score = cross_val_score(est, X, y, cv=inner, scoring="roc_auc").mean()
        out[name] = (score, est.fit(X, y).predict_proba(test.reshape(-1, 1))[:, 1])
    return out


t0 = time.time()
_res = Parallel(n_jobs=N_JOBS)(delayed(run_fold)(s["test"], LABELS, SEED + 100 * s["repeat"] + s["fold"])
                               for s in FOLDS)
Q = {n: np.full((N_REPEATS, len(subjects)), np.nan) for n in CANDIDATES + ["selected"]}
picks = []
for s, out in zip(FOLDS, _res):
    best = max(CANDIDATES, key=lambda n: out[n][0])      # ties -> the earlier, simpler candidate
    for n in CANDIDATES:
        Q[n][s["repeat"] - 1, s["test"]] = out[n][1]
    Q["selected"][s["repeat"] - 1, s["test"]] = out[best][1]
    picks.append(dict(repeat=s["repeat"], fold=s["fold"], selected=best,
                      **{f"inner_auc {n}": round(out[n][0], 4) for n in CANDIDATES}))
PICKS = pd.DataFrame(picks)
assert all(np.isfinite(q).all() for q in Q.values())
print(f"{len(FOLDS)} outer folds in {time.time() - t0:.0f} s; selected per fold:")
print(PICKS.selected.value_counts().to_string())


25 outer folds in 268 s; selected per fold:
selected
C3 lagged cross-correlation    13
C2 Riemannian covariance        5
C5 soft vote                    3
C4 parameterised spectrum       3
C1 band power                   1


In [7]:
# ===========================================================================
# CELL 7 — AD VS FTD: EVERY CANDIDATE, WITH CIs AND PAIRED TESTS
# ===========================================================================
YD = (LABELS[DEMENTIA] == AD).astype(int)
POS, NEG = np.flatnonzero(YD == 1), np.flatnonzero(YD == 0)


def auc(pos_scores, neg_scores):
    """Mann-Whitney AUC = P(an AD score > an FTD score), ties count one half. Works on stacked rows."""
    d = pos_scores[..., :, None] - neg_scores[..., None, :]
    return (d > 0).mean((-1, -2)) + 0.5 * (d == 0).mean((-1, -2))


def per_repeat(q):
    qd = q[:, DEMENTIA]
    call = qd >= 0.5
    rec_ad, rec_ftd = call[:, POS].mean(1), (~call[:, NEG]).mean(1)
    return pd.DataFrame(dict(auc=auc(qd[:, POS], qd[:, NEG]), bal_acc=(rec_ad + rec_ftd) / 2,
                             recall_AD=rec_ad, recall_FTD=rec_ftd))


_rng = np.random.default_rng(SEED)
_bp = _rng.choice(POS, (N_BOOT, len(POS)))               # each class resampled separately
_bn = _rng.choice(NEG, (N_BOOT, len(NEG)))
BOOT = {}
for n, q in Q.items():
    qd = q[:, DEMENTIA]                                   # (R, 59)
    BOOT[n] = np.stack([auc(qd[r][_bp], qd[r][_bn]) for r in range(N_REPEATS)]).mean(0)   # repeat mean per resample


def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); out = np.empty_like(p); run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - k) * p[i])); out[i] = run
    return out


rows = []
for n, q in Q.items():
    pr = per_repeat(q)
    d = BOOT[n] - BOOT["C1 band power"]
    rows.append(dict(candidate=n, auc=pr.auc.mean(), auc_lo=np.percentile(BOOT[n], 2.5),
                     auc_hi=np.percentile(BOOT[n], 97.5), auc_sd_repeats=pr.auc.std(ddof=1),
                     bal_acc=pr.bal_acc.mean(), recall_AD=pr.recall_AD.mean(), recall_FTD=pr.recall_FTD.mean(),
                     d_auc_vs_bp=pr.auc.mean() - per_repeat(Q["C1 band power"]).auc.mean(),
                     d_lo=np.percentile(d, 2.5), d_hi=np.percentile(d, 97.5),
                     p_vs_bp=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())) if n != "C1 band power" else np.nan,
                     times_selected=int((PICKS.selected == n).sum())))
RESULTS = pd.DataFrame(rows).set_index("candidate")
_fam = ["C2 Riemannian covariance", "C3 lagged cross-correlation", "C4 parameterised spectrum", "C5 soft vote"]
RESULTS.loc[_fam, "p_holm"] = holm(RESULTS.loc[_fam, "p_vs_bp"])

pd.set_option("display.width", 200)
print("AD vs FTD, 59 patients, 5 x 5-fold (the 3-class folds); AUC = P(AD scores higher); mean over repeats\n")
print(RESULTS[["auc", "auc_lo", "auc_hi", "auc_sd_repeats", "bal_acc", "recall_AD", "recall_FTD",
               "d_auc_vs_bp", "d_lo", "d_hi", "p_vs_bp", "p_holm", "times_selected"]].round(3).to_string())
print("\ncomparators: 3-class GAT 0.595; aperiodic scores 0.616-0.644 (Experiment 8); "
      "Mlinaric et al. 2026 0.651 (LOSO); target >= 0.70")


AD vs FTD, 59 patients, 5 x 5-fold (the 3-class folds); AUC = P(AD scores higher); mean over repeats

                               auc  auc_lo  auc_hi  auc_sd_repeats  bal_acc  recall_AD  recall_FTD  d_auc_vs_bp   d_lo   d_hi  p_vs_bp  p_holm  times_selected
candidate                                                                                                                                                     
C1 band power                0.524   0.405   0.634           0.040    0.486      0.572       0.400        0.000  0.000  0.000      NaN     NaN               1
C2 Riemannian covariance     0.549   0.418   0.674           0.032    0.548      0.644       0.452        0.025 -0.105  0.164    0.734   1.000               5
C3 lagged cross-correlation  0.628   0.502   0.748           0.038    0.605      0.722       0.487        0.104 -0.033  0.246    0.139   0.556              13
C4 parameterised spectrum    0.542   0.403   0.677           0.033    0.552      0.617       0.487     

In [8]:
# ===========================================================================
# CELL 8 — CONFOUNDS: SEVERITY, DEMOGRAPHICS AND THE ICA RANK
# ===========================================================================
_s = subjects.iloc[DEMENTIA]
COVARS = {"MMSE (lower = AD)": -_s.mmse.to_numpy(float), "age": _s.age.to_numpy(float),
          "sex (female = AD)": (_s.sex == "F").to_numpy(float), "ICA rank deficit": 19.0 - RANK[DEMENTIA]}
single = {k: auc(v[POS], v[NEG]) for k, v in COVARS.items()}


def loglik(Xc, y):
    m = LogisticRegression(C=np.inf, max_iter=10000).fit(Xc, y)          # unpenalised
    p = np.clip(m.predict_proba(Xc)[:, 1], 1e-12, 1 - 1e-12)
    return (y * np.log(p) + (1 - y) * np.log(1 - p)).sum()


_base = StandardScaler().fit_transform(np.column_stack([_s.mmse, _s.age, _s.sex == "F"]).astype(float))
_ll0 = loglik(_base, YD)
conf = []
for n, q in Q.items():
    qm = np.clip(q[:, DEMENTIA].mean(0), 1e-6, 1 - 1e-6)            # each patient's out-of-fold q, repeat mean
    lq = np.log(qm / (1 - qm))
    lr = 2 * (loglik(np.column_stack([_base, (lq - lq.mean()) / lq.std()]), YD) - _ll0)
    conf.append(dict(candidate=n, lr_chi2=lr, p_beyond_mmse_age_sex=stats.chi2.sf(lr, 1),
                     r_with_mmse=stats.spearmanr(qm, _s.mmse)[0],
                     r_with_rank_deficit=stats.spearmanr(qm, COVARS["ICA rank deficit"])[0]))
CONFOUNDS = pd.DataFrame(conf).set_index("candidate")
print("single non-EEG variables, AD vs FTD AUC: " + ", ".join(f"{k} {v:.3f}" for k, v in single.items()))
print("\ndoes each score add to MMSE + age + sex? (likelihood-ratio test, 1 df)\n")
print(CONFOUNDS.round(3).to_string())


single non-EEG variables, AD vs FTD AUC: MMSE (lower = AD) 0.806, age 0.574, sex (female = AD) 0.638, ICA rank deficit 0.558

does each score add to MMSE + age + sex? (likelihood-ratio test, 1 df)

                             lr_chi2  p_beyond_mmse_age_sex  r_with_mmse  r_with_rank_deficit
candidate                                                                                    
C1 band power                  1.449                  0.229       -0.143               -0.179
C2 Riemannian covariance       0.117                  0.732       -0.262                0.162
C3 lagged cross-correlation    0.460                  0.498       -0.361                0.229
C4 parameterised spectrum      0.017                  0.898       -0.127               -0.189
C5 soft vote                   0.019                  0.891       -0.304                0.214
selected                       0.003                  0.959       -0.368                0.268


In [9]:
# ===========================================================================
# CELL 9 — CHANCE LEVEL: THE WHOLE NESTED PIPELINE ON SHUFFLED DIAGNOSES (repeat 1)
# ===========================================================================
_r1 = [s for s in FOLDS if s["repeat"] == 1]


def shuffled_auc(k):
    labels = LABELS.copy()
    labels[DEMENTIA] = np.random.default_rng(SEED + 10_000 + k).permutation(labels[DEMENTIA])
    q = np.full(len(labels), np.nan)
    for s in _r1:
        out = run_fold(s["test"], labels, SEED + 100 + s["fold"])
        q[s["test"]] = out[max(CANDIDATES, key=lambda n: out[n][0])][1]
    yd = labels[DEMENTIA] == AD
    return auc(q[DEMENTIA][yd], q[DEMENTIA][~yd])


t0 = time.time()
NULL = np.array(Parallel(n_jobs=N_JOBS)(delayed(shuffled_auc)(k) for k in range(N_PERM))) if N_PERM else np.array([])
_obs = per_repeat(Q["selected"][:1]).auc.iloc[0]
P_PERM = (1 + (NULL >= _obs).sum()) / (1 + len(NULL)) if len(NULL) else np.nan
print(f"{len(NULL)} shuffles in {time.time() - t0:.0f} s; selected pipeline, repeat 1: AUC {_obs:.3f}")
if len(NULL):
    print(f"shuffled AUCs: median {np.median(NULL):.3f}, 95 % range {np.percentile(NULL, 2.5):.3f}-"
          f"{np.percentile(NULL, 97.5):.3f}; one-sided p = {P_PERM:.3f}")


100 shuffles in 3401 s; selected pipeline, repeat 1: AUC 0.528
shuffled AUCs: median 0.479, 95 % range 0.291-0.651; one-sided p = 0.337


In [10]:
# ===========================================================================
# CELL 10 — SAVE
# ===========================================================================
np.savez_compressed(OUT_DIR / "stage2_oof.npz", subjects=subjects.participant_id.to_numpy(), labels=LABELS,
                    candidates=np.array(list(Q)), q_ad=np.stack([Q[n] for n in Q]),
                    fold_of=np.stack([np.concatenate([np.full(len(s["test"]), s["fold"]) for s in FOLDS
                                                      if s["repeat"] == r])[np.argsort(np.concatenate(
                                                          [s["test"] for s in FOLDS if s["repeat"] == r]))]
                                      for r in range(1, N_REPEATS + 1)]))
RESULTS.to_csv(OUT_DIR / "candidates.csv")
CONFOUNDS.to_csv(OUT_DIR / "confounds.csv")
PICKS.to_csv(OUT_DIR / "selection.csv", index=False)
pd.DataFrame(dict(shuffled_auc=NULL)).to_csv(OUT_DIR / "permutation.csv", index=False)
_sel = RESULTS.loc["selected"]
SUMMARY = dict(task="AD vs FTD, 36 vs 23, stage 2 of the two-stage plan",
               protocol=PROTOCOL["written"], folds=f"{N_REPEATS} x {N_FOLDS}-fold, the 3-class notebook's folds",
               selected=dict(auc=round(_sel.auc, 4), ci=[round(_sel.auc_lo, 4), round(_sel.auc_hi, 4)],
                             bal_acc=round(_sel.bal_acc, 4), recall_AD=round(_sel.recall_AD, 4),
                             recall_FTD=round(_sel.recall_FTD, 4),
                             picks=PICKS.selected.value_counts().to_dict(),
                             p_vs_band_power=round(_sel.p_vs_bp, 4), p_permutation_repeat1=P_PERM,
                             p_beyond_mmse_age_sex=round(CONFOUNDS.loc["selected", "p_beyond_mmse_age_sex"], 4)),
               candidates={n: dict(auc=round(r.auc, 4), ci=[round(r.auc_lo, 4), round(r.auc_hi, 4)],
                                   p_holm_vs_band_power=None if pd.isna(r.p_holm) else round(r.p_holm, 4))
                           for n, r in RESULTS.drop(index="selected").iterrows()},
               non_eeg_auc={k: round(v, 4) for k, v in single.items()},
               target="AD vs FTD AUC >= 0.70; best honest published 0.651 (Mlinaric et al. 2026)")
(OUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=float),
                                      encoding="utf-8")
print(f"saved to {OUT_DIR}:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


saved to C:\Projects\Major Project\experiments_ad_vs_ftd\results\riemann: candidates.csv, confounds.csv, permutation.csv, protocol.json, selection.csv, stage2_oof.npz, summary.json
